# Benchmark mémoire COSMOS — automatique

**Avant de lancer** (panneau de droite) :
- **Session options → Accelerator : GPU T4 x2**
- **Session options → Internet : On** (téléchargement des modèles)
- **Add Input** : ton Dataset contenant le dossier COSMOS (`cosmos/` + `benchmark/`)

**Exécute les cellules dans l'ordre** (ou *Run All*).
Pour tout lancer **en arrière-plan** : *Save Version → Save & Run All (Commit)*, puis tu peux fermer l'onglet.

**À la fin**, dans l'onglet **Output** du notebook :
- `benchmark-cosmos.zip` : tout (rapport, graphiques, tableaux CSV, résultats bruts, logs)
- `report.html` : le rapport seul, à ouvrir dans le navigateur

In [ ]:
# 1) Diagnostic de la machine
!nvidia-smi -L
!python --version
!df -h /kaggle/working | tail -1
!df -h / | tail -1

In [ ]:
# 2) Récupère le code COSMOS depuis le Dataset (Kaggle décompresse souvent le zip tout seul)
import glob, os, shutil, zipfile
dst = '/kaggle/working/cosmos'
shutil.rmtree(dst, ignore_errors=True)
hits = glob.glob('/kaggle/input/**/benchmark/run_all.py', recursive=True)
if hits:
    shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), dst)
else:
    zips = glob.glob('/kaggle/input/**/cosmos_bench.zip', recursive=True)
    assert zips, 'Rien trouve dans /kaggle/input : clique sur Add Input et ajoute le Dataset'
    zipfile.ZipFile(zips[0]).extractall(dst)
%cd /kaggle/working/cosmos
!ls /kaggle/working/cosmos /kaggle/working/cosmos/benchmark

In [ ]:
# 3) Dépendances + vérification
!pip install -q fastembed
!pip install -q -U transformers accelerate
!python -c "import fastembed, transformers, torch, matplotlib, pandas; print('transformers', transformers.__version__, '| torch', torch.__version__, '| gpus', torch.cuda.device_count())"

## 4) Lancer le benchmark

Le benchmark complet fait, dans l'ordre :
1. test minuscule (Qwen3-0.6B) : arrêt immédiat s'il échoue
2. COSMOS seul, sans IA, de 50 à 1600 souvenirs de bruit
3. 7 modèles de 1,7 à 4 milliards de paramètres, deux en parallèle (un par GPU) :
   Qwen3.5-4B, Gemma-3-4B (Google), Qwen3-4B, Phi-4-mini (Microsoft), Llama-3.2-3B (Meta), SmolLM3-3B (Hugging Face), Qwen3-1.7B
4. 1 gros modèle sur les deux GPU : Qwen3-8B
5. rapport (graphiques + tableaux) et `benchmark-cosmos.zip`

Durée estimée : 7 à 9 h. Aucun nouveau modèle après 8 h, arrêt forcé à 10 h 30 (limite Kaggle : 12 h).
Les résultats sont sauvegardés après chaque série : même un modèle coupé laisse des résultats partiels.

**`QUICK = True`** → test rapide (≈ 10 min, un seul petit modèle) pour vérifier que tout marche.
**`QUICK = False`** → benchmark complet.

In [ ]:
QUICK = False   # True = test rapide ; False = benchmark complet (à lancer avec Save & Run All)

import os
if not os.path.isdir('/kaggle/working/cosmos/benchmark'):
    raise SystemExit("Exécute d'abord les cellules 1 à 3 (ou utilise 'Run All').")
%cd /kaggle/working/cosmos
args = "--models Qwen/Qwen3-1.7B --scales 50 --seeds 0 --synthetic 10 --limit 20" if QUICK else ""
!python -m benchmark.run_all $args

## 5) Résultats
Regénère le rapport (au cas où le run aurait été interrompu), puis l'affiche ici.

In [ ]:
import glob, os, shutil
base = '/kaggle/working/cosmos/benchmark/results'
!cd /kaggle/working/cosmos && python -m benchmark.report --results {base} --zip /kaggle/working/benchmark-cosmos.zip
if os.path.exists(base + '/report.html'):
    shutil.copy(base + '/report.html', '/kaggle/working/report.html')

In [ ]:
from IPython.display import display, Markdown, Image
p = base + '/SUMMARY.md'
if os.path.exists(p):
    display(Markdown(open(p, encoding='utf-8').read()))
for img in sorted(glob.glob(base + '/charts/*.png')):
    display(Image(img))
if not os.path.exists(p):
    print('Pas de résumé : lis la sortie de la cellule 4.')

In [ ]:
# 6) Contenu du zip final
import zipfile
z = '/kaggle/working/benchmark-cosmos.zip'
if os.path.exists(z):
    print(z, round(os.path.getsize(z) / 1e6, 1), 'Mo')
    for n in zipfile.ZipFile(z).namelist():
        print(' ', n)
else:
    print('Zip absent : lis les erreurs ci-dessus.')